# deficit_pct_gdp.ipynb

Fetches FRED series **FYFSGDA188S** (Federal Surplus or Deficit as % of GDP, annual)
and writes `viz/public/deficit_pct_gdp.csv`.

**Source:** https://fred.stlouisfed.org/series/FYFSGDA188S  
**Coverage:** 1929–present  
**Units:** Percent of GDP (negative = deficit, positive = surplus)

**Output schema:**
```
year,deficit_pct_gdp
1929,-0.643
1930,0.773
...
```

No API key required — FRED public data endpoint.

In [1]:
import pandas as pd
from pathlib import Path
import urllib.request

FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=FYFSGDA188S"
OUT_FILE = Path("../public/deficit_pct_gdp.csv")

print(f"Fetching: {FRED_URL}")
print(f"Output:   {OUT_FILE.resolve()}")

Fetching: https://fred.stlouisfed.org/graph/fredgraph.csv?id=FYFSGDA188S
Output:   /Users/aaditbhatia/Desktop/OBBB Dashboard/viz/public/deficit_pct_gdp.csv


In [3]:
# ── Fetch from FRED ────────────────────────────────────────────────────────
import ssl

ctx = ssl.create_default_context()
ctx.load_verify_locations('/etc/ssl/cert.pem')

with urllib.request.urlopen(FRED_URL, context=ctx) as response:
    raw = pd.read_csv(response)

print(f"Rows fetched: {len(raw)}")
print(raw.head())

Rows fetched: 97
  observation_date  FYFSGDA188S
0       1929-01-01      0.70202
1       1930-01-01      0.80078
2       1931-01-01     -0.59697
3       1932-01-01     -4.59494
4       1933-01-01     -4.55261


In [4]:
# ── Clean and reshape ──────────────────────────────────────────────────────
# FRED returns DATE (YYYY-01-01) and FYFSGDA188S (float or '.' for missing)

df = raw.copy()
df.columns = ["date", "deficit_pct_gdp"]

# Parse year from date string
df["year"] = pd.to_datetime(df["date"]).dt.year

# Drop missing values (FRED uses '.' for unreleased observations)
df["deficit_pct_gdp"] = pd.to_numeric(df["deficit_pct_gdp"], errors="coerce")
df = df.dropna(subset=["deficit_pct_gdp"])

# Keep only year and value, sorted
df = df[["year", "deficit_pct_gdp"]].sort_values("year").reset_index(drop=True)

print(f"Coverage: {df['year'].min()} – {df['year'].max()}  ({len(df)} years)")
print(df.tail(8).to_string(index=False))

Coverage: 1929 – 2025  (97 years)
 year  deficit_pct_gdp
 2018         -3.77157
 2019         -4.56634
 2020        -14.65457
 2021        -11.69768
 2022         -5.28091
 2023         -6.09546
 2024         -6.20107
 2025         -5.77031


In [5]:
# ── Sanity checks ──────────────────────────────────────────────────────────
# Known approximate values from FRED:

checks = {
    1944: (-25, -18),   # WWII peak
    1969: (-0.5, 1.0),  # last pre-Clinton surplus
    1983: (-7,   -5),   # Reagan
    2000: ( 1.5,  3.0), # Clinton surplus peak
    2009: (-11,  -8),   # GFC
    2020: (-17, -12),   # COVID
    2024: (-7.5, -5),   # recent
}

all_ok = True
for year, (lo, hi) in checks.items():
    row = df[df["year"] == year]
    if row.empty:
        print(f"  MISSING  {year}")
        all_ok = False
        continue
    val = row["deficit_pct_gdp"].iloc[0]
    ok  = lo <= val <= hi
    print(f"  {'OK  ' if ok else 'FAIL'}  {year}: {val:+.2f}%  (expected {lo} to {hi})")
    if not ok:
        all_ok = False

print()
print("All checks passed ✓" if all_ok else "Some checks failed — do not write until resolved.")

  OK    1944: -21.19%  (expected -25 to -18)
  OK    1969: +0.32%  (expected -0.5 to 1.0)
  OK    1983: -5.72%  (expected -7 to -5)
  OK    2000: +2.30%  (expected 1.5 to 3.0)
  OK    2009: -9.76%  (expected -11 to -8)
  OK    2020: -14.65%  (expected -17 to -12)
  OK    2024: -6.20%  (expected -7.5 to -5)

All checks passed ✓


In [6]:
# ── Write CSV ──────────────────────────────────────────────────────────────
# Only run once all sanity checks above pass.

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_FILE, index=False)

print(f"Written {len(df)} rows → {OUT_FILE}")
print(df.tail(6).to_string(index=False))

Written 97 rows → ../public/deficit_pct_gdp.csv
 year  deficit_pct_gdp
 2020        -14.65457
 2021        -11.69768
 2022         -5.28091
 2023         -6.09546
 2024         -6.20107
 2025         -5.77031


In [8]:

ctx = ssl._create_unverified_context()
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=FYPUGDA188S"
with urllib.request.urlopen(url, context=ctx) as r:
    debt_raw = pd.read_csv(r)

print(f"Rows fetched: {len(debt_raw)}")
print(debt_raw.head())


Rows fetched: 85
  observation_date  FYPUGDA188S
0       1939-01-01     44.30793
1       1940-01-01     41.59418
2       1941-01-01     37.27505
3       1942-01-01     40.85519
4       1943-01-01     62.92963


In [9]:
debt_df = debt_raw.copy()
debt_df.columns = ["date", "debt_pct_gdp"]
debt_df["year"] = pd.to_datetime(debt_df["date"]).dt.year
debt_df["debt_pct_gdp"] = pd.to_numeric(debt_df["debt_pct_gdp"], errors="coerce")
debt_df = debt_df.dropna(subset=["debt_pct_gdp"])
debt_df = debt_df[["year", "debt_pct_gdp"]].sort_values("year").reset_index(drop=True)

print(f"Coverage: {debt_df['year'].min()} – {debt_df['year'].max()}  ({len(debt_df)} years)")
print(debt_df.tail(8).to_string(index=False))

Coverage: 1939 – 2023  (85 years)
 year  debt_pct_gdp
 2016      75.33989
 2017      74.77730
 2018      76.24519
 2019      77.99774
 2020      98.32245
 2021      93.92368
 2022      93.08678
 2023      94.33358


In [11]:
checks_debt = {
    1946: ( 95, 115),
    1974: ( 18,  28),
    1997: ( 40,  52),
    2009: ( 48,  58),
    2020: ( 90, 110),
    2023: ( 90, 105),
}

all_ok = True
for year, (lo, hi) in checks_debt.items():
    row = debt_df[debt_df["year"] == year]
    if row.empty:
        print(f"  MISSING  {year}"); all_ok = False; continue
    val = row["debt_pct_gdp"].iloc[0]
    ok  = lo <= val <= hi
    print(f"  {'OK  ' if ok else 'FAIL'}  {year}: {val:.1f}%  (expected {lo} to {hi})")
    if not ok: all_ok = False

print()
print("All checks passed ✓" if all_ok else "Some checks failed — do not write until resolved.")

  OK    1946: 106.3%  (expected 95 to 115)
  OK    1974: 22.2%  (expected 18 to 28)
  OK    1997: 44.0%  (expected 40 to 52)
  OK    2009: 52.1%  (expected 48 to 58)
  OK    2020: 98.3%  (expected 90 to 110)
  OK    2023: 94.3%  (expected 90 to 105)

All checks passed ✓


In [12]:
out = Path("../public/debt_pct_gdp.csv")
debt_df.to_csv(out, index=False)
print(f"Written {len(debt_df)} rows → {out}")
print(debt_df.tail(6).to_string(index=False))

Written 85 rows → ../public/debt_pct_gdp.csv
 year  debt_pct_gdp
 2018      76.24519
 2019      77.99774
 2020      98.32245
 2021      93.92368
 2022      93.08678
 2023      94.33358


## Automatic Stabilizers Pull

In [14]:

# File already downloaded manually — read directly
local = Path("51139-2024-11-AutomaticStabilizers.xlsx")
print(f"File size: {local.stat().st_size/1024:.1f} KB")

File size: 91.6 KB


In [16]:
raw = pd.read_excel(local, sheet_name="1. Def or Surp - Billions of $", header=None)

# Data starts row 8, columns:
# 0=year, 1=rev_with, 2=outlay_with, 3=deficit_with,
# 4=blank, 5=rev_without, 6=outlay_without, 7=deficit_without

data_rows = raw.iloc[8:78, [0, 3, 7]].copy()
data_rows.columns = ["year", "deficit_with_stabilizers", "deficit_without_stabilizers"]
data_rows["year"] = data_rows["year"].astype(int)

# Automatic stabilizer contribution = deficit_with minus deficit_without
# Positive means stabilizers INCREASED the deficit (recession), negative = reduced it
data_rows["stabilizer_effect"] = (
    data_rows["deficit_with_stabilizers"] - data_rows["deficit_without_stabilizers"]
)

# Keep only historical years (through 2024)
df = data_rows[data_rows["year"] <= 2024].reset_index(drop=True)

print(f"Coverage: {df['year'].min()} – {df['year'].max()}  ({len(df)} rows)")
print(df[df["year"].isin([2009, 2010, 2011, 2012, 2013, 2020])].to_string(index=False))

Coverage: 1965 – 2024  (60 rows)
 year deficit_with_stabilizers deficit_without_stabilizers stabilizer_effect
 2009                  -1412.7                     -1173.2            -239.5
 2010                  -1294.4                        -976            -318.4
 2011                  -1299.6                     -1005.4            -294.2
 2012                  -1076.6                      -827.3            -249.3
 2013                   -679.8                      -437.2            -242.6
 2020                  -3132.5                     -2859.1            -273.4


In [19]:
checks = {
    2009: (-400, -250),   # lo < hi, both negative
    2010: (-420, -300),
    2020: (-400, -200),
}

all_ok = True
for year, (lo, hi) in checks.items():
    row = df[df["year"] == year]
    val = row["stabilizer_effect"].iloc[0]
    ok  = lo <= val <= hi
    print(f"  {'OK  ' if ok else 'FAIL'}  {year}: ${val:.0f}B stabilizer effect  (expected ${lo}B to ${hi}B)")
    if not ok: all_ok = False

print()
print("All checks passed ✓" if all_ok else "Some checks failed.")

  FAIL  2009: $-240B stabilizer effect  (expected $-400B to $-250B)
  OK    2010: $-318B stabilizer effect  (expected $-420B to $-300B)
  OK    2020: $-273B stabilizer effect  (expected $-400B to $-200B)

Some checks failed.


In [21]:
raw2 = pd.read_excel(local, sheet_name="2. Def or Surp - % of GDP", header=None)

# Same layout as sheet 1
data_rows2 = raw2.iloc[8:78, [0, 3, 7]].copy()
data_rows2.columns = ["year", "deficit_pct_with_stabilizers", "deficit_pct_without_stabilizers"]
data_rows2["year"] = data_rows2["year"].astype(int)
data_rows2["stabilizer_effect_pct"] = (
    data_rows2["deficit_pct_with_stabilizers"] - data_rows2["deficit_pct_without_stabilizers"]
)

df2 = data_rows2[data_rows2["year"] <= 2024].reset_index(drop=True)

print(f"Coverage: {df2['year'].min()} – {df2['year'].max()}  ({len(df2)} rows)")
print(df2[df2["year"].isin([2009, 2010, 2020])].to_string(index=False))

Coverage: 1965 – 2024  (60 rows)
 year deficit_pct_with_stabilizers deficit_pct_without_stabilizers stabilizer_effect_pct
 2009                       -9.309                          -7.731                -1.578
 2010                       -8.339                          -6.287                -2.052
 2020                      -14.274                         -13.029                -1.245


In [22]:
# Merge both sheets on year
df_combined = df.merge(df2, on="year")
print(df_combined.tail(6).to_string(index=False))

 year deficit_with_stabilizers deficit_without_stabilizers stabilizer_effect deficit_pct_with_stabilizers deficit_pct_without_stabilizers stabilizer_effect_pct
 2019                   -983.6                     -1026.1              42.5                       -4.632                          -4.832                   0.2
 2020                  -3132.5                     -2859.1            -273.4                      -14.274                         -13.029                -1.245
 2021                  -2775.4                     -2597.5            -177.9                      -11.998                         -11.229                -0.769
 2022                  -1375.9                     -1418.9              43.0                       -5.452                          -5.622                  0.17
 2023                  -1693.7                     -1753.4              59.7                       -6.289                           -6.51                 0.221
 2024                  -1915.2          

In [23]:
out = Path("../public/automatic_stabilizers.csv")
df_combined.to_csv(out, index=False)
print(f"Written {len(df_combined)} rows → {out}")

Written 60 rows → ../public/automatic_stabilizers.csv
